In [ ]:
#
# use with godot/arm3_no_physics
#
import sys
sys.path.append("../../")

from lib.data.dataplot import *
from lib.dds.dds import *
from lib.utils.time import *
from lib.system.controllers import *
from lib.system.trajectory import *
from lib.system.manipulator import *

class ManipulatorRobot:

    def __init__(self):
        self.arm = ThreeJointsPlanarArm(0.6, 0.58, 0.056,
                                        0.5, 0.5, 0.5,
                                        0.8)

        # joint 1
        self.speed_control_1 = PID_Controller(80, 120, 0,
                                              20)  # 20Nm max torque, antiwindup

        # joint 2
        self.speed_control_2 = PID_Controller(40, 20, 0,
                                              20)  # 20Nm max torque, antiwindup

        # joint 3
        self.speed_control_3 = PID_Controller(3, 4.0, 0,
                                              20)  # 20Nm max torque, antiwindup

        self.pos_control_1 = PID_Controller(10, 0, 0, 10)  # 10 rad/s max speed
        self.pos_control_2 = PID_Controller(10, 0, 0, 10)  # 10 rad/s max speed
        self.pos_control_3 = PID_Controller(10, 0, 0, 10)  # 10 rad/s max speed
        
        
    def start(self, x_target, y_target, a_target):
        self.trajectory = StraightLineMotion(0.2, 1.0, 1.0, 2)
        (xc, yc, ac) = self.get_pose()
        self.trajectory.start_motion( (xc, yc, ac), (x_target, y_target, a_target))

        
    def evaluate(self, delta_t):
        
        (x, y, a) = self.trajectory.evaluate(delta_t)

        (self.theta1, self.theta2, self.theta3) = self.arm.inverse_kinematics(x, y, a)
        
        self.wref_1 = self.pos_control_1.evaluate(delta_t, self.theta1 - self.arm.element_1.theta)
        self.wref_2 = self.pos_control_2.evaluate(delta_t, self.theta2 - self.arm.element_2.theta)
        self.wref_3 = self.pos_control_3.evaluate(delta_t, self.theta3 - self.arm.element_3.theta)
        
        torque1 = self.speed_control_1.evaluate(delta_t, self.wref_1 - self.arm.element_1.w)
        torque2 = self.speed_control_2.evaluate(delta_t, self.wref_2 - self.arm.element_2.w)
        torque3 = self.speed_control_3.evaluate(delta_t, self.wref_3 - self.arm.element_3.w)

        self.arm.evaluate(delta_t, torque1, torque2, torque3)

    def get_joint_angles(self):
        return self.arm.get_joint_angles()
    
    def get_pose(self):
        return self.arm.get_pose()


dds = DDS()
dds.start()

dds.subscribe(['tick','target_x','target_y','target_alpha','sequence'])

robot = ManipulatorRobot()
robot.start(0.8, 0.1, math.radians(-90))

t = Time()
t.start()
current_sequence = dds.read('sequence')
while True: #t.get() < 10:
    
    dds.wait('tick')    
    delta_t = t.elapsed()
    
    robot.evaluate(delta_t)
    (t1, t2, t3) = robot.get_joint_angles()
    (x, y, a) = robot.get_pose()

    dds.publish('theta1', t1, DDS.DDS_TYPE_FLOAT)
    dds.publish('theta2', t2, DDS.DDS_TYPE_FLOAT)
    dds.publish('theta3', t3, DDS.DDS_TYPE_FLOAT)
    dds.publish('x', x, DDS.DDS_TYPE_FLOAT)
    dds.publish('y', y, DDS.DDS_TYPE_FLOAT)
    dds.publish('a', a, DDS.DDS_TYPE_FLOAT)
    
    sequence = dds.read('sequence')
    if sequence != current_sequence:
        current_sequence = sequence
        target_x = dds.read('target_x')
        target_y = dds.read('target_y')
        target_alpha = dds.read('target_alpha')
        robot.start(target_x, target_y, math.radians(target_alpha))

dds.stop()

